### Install required libraries

In [6]:
!pip install -q gradio pymupdf sentence-transformers numpy

In [7]:
# Install llama-cpp-python with CUDA 12.x support
!pip install --no-cache-dir llama-cpp-python==0.2.90 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu123

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu123
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.5/444.5 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 168.6 MB/s eta 0:00:00


### Import libraries

In [8]:
import os
import json
import fitz
import gradio as gr
import numpy as np
from datetime import datetime
from sentence_transformers import SentenceTransformer
from llama_cpp import Llama

### Load open-source models

In [9]:
model_path = "/content/mistral.gguf"  # change this to your model path
if not os.path.exists(model_path):
    !wget https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf -O {model_path}
    print(f"Model downloaded to {model_path}")

# Verify file exists and check size
if os.path.exists(model_path):
    print(f"Model file exists. Size: {os.path.getsize(model_path) / (1024 * 1024):.2f} MB")
else:
    print("Model file not found!")

# Load the model with GPU acceleration
try:
    llm = Llama(
        model_path=model_path,
        n_gpu_layers=1,  # Start with 1 layer on GPU to be safe
        n_ctx=2048,      # Context window size
        verbose=True     # Show loading progress
    )

    print("Model loaded successfully!")



except Exception as e:
    print(f"Error loading model: {e}")

--2026-07-13 08:44:37--  https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf
Resolving huggingface.co (huggingface.co)... 18.164.174.17, 18.164.174.55, 18.164.174.118, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.17|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/65778ac662d3ac1817cc9201/865f5e4682dddb29c2e20270b2471a7590c83a414bbf1d72cf4c08fdff2eeca4?X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27mistral-7b-instruct-v0.2.Q4_K_M.gguf%3B+filename%3D%22mistral-7b-instruct-v0.2.Q4_K_M.gguf%22%3B&user_id=public&Expires=1783935877&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjU3NzhhYzY2MmQzYWMxODE3Y2M5MjAxLzg2NWY1ZTQ2ODJkZGRiMjljMmUyMDI3MGIyNDcxYTc1OTBjODNhNDE0YmJmMWQ3MmNmNGMwOGZkZmYyZWVjYTRcXD9YLVhldC1DYXMtVWlkPXB1YmxpYyZyZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSo

llama_model_loader: loaded meta data with 24 key-value pairs and 291 tensors from /content/mistral.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.2
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv   7:                 llama.attention.

Model loaded successfully!


AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | AVX512_BF16 = 0 | FMA = 1 | NEON = 0 | SVE = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | MATMUL_INT8 = 0 | LLAMAFILE = 1 | 
Model metadata: {'tokenizer.chat_template': "{{ bos_token }}{% for message in messages %}{% if (message['role'] == 'user') != (loop.index0 % 2 == 0) %}{{ raise_exception('Conversation roles must alternate user/assistant/user/assistant/...') }}{% endif %}{% if message['role'] == 'user' %}{{ '[INST] ' + message['content'] + ' [/INST]' }}{% elif message['role'] == 'assistant' %}{{ message['content'] + eos_token}}{% else %}{{ raise_exception('Only user and assistant roles are supported!') }}{% endif %}{% endfor %}", 'tokenizer.ggml.add_eos_token': 'false', 'tokenizer.ggml.padding_token_id': '0', 'tokenizer.ggml.unknown_token_id': '0', 'tokenizer.ggml.eos_token_id': '2', 'general.architecture': 'llama', 'llama.rope.freq_base': 

### Create Mistral helper function

In [10]:
def mistral_model(prompt, max_tokens=300, temperature=0.0):
    formatted_prompt = f"""
[INST]
{prompt}
[/INST]
"""

    response = llm(
        formatted_prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        stop=["</s>", "[/INST]"]
    )

    return response["choices"][0]["text"].strip()

### Load embedding model

In [11]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.


### Extract text from PDFs

In [12]:
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []

    for page_num, page in enumerate(doc, start=1):
        text = page.get_text("text")

        pages.append({
            "filename": os.path.basename(pdf_path),
            "page_number": page_num,
            "text": text
        })

    doc.close()
    return pages

### Chunk PDF text

In [13]:
def chunk_text(text, chunk_size=800, overlap=150):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]

        if chunk.strip():
            chunks.append(chunk.strip())

        start += chunk_size - overlap

    return chunks

### Build searchable document index

In [14]:
document_chunks = []
chunk_embeddings = None

def build_index(pdf_files):
    global document_chunks, chunk_embeddings

    if not pdf_files:
        return "Please upload at least one PDF."

    document_chunks = []

    for pdf_path in pdf_files:
        pages = extract_text_from_pdf(pdf_path)

        for page in pages:
            chunks = chunk_text(page["text"])

            for chunk_index, chunk in enumerate(chunks):
                document_chunks.append({
                    "filename": page["filename"],
                    "page_number": page["page_number"],
                    "chunk_index": chunk_index,
                    "text": chunk
                })

    if len(document_chunks) == 0:
        return "No readable text was found in the uploaded PDFs."

    texts = [chunk["text"] for chunk in document_chunks]

    chunk_embeddings = embedding_model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    return f"Indexed {len(pdf_files)} PDF(s) and created {len(document_chunks)} searchable chunks."

### Retrieve relevant chunks

In [15]:
def retrieve_relevant_chunks(query, top_k=3):
    global chunk_embeddings, document_chunks

    if chunk_embeddings is None or len(document_chunks) == 0:
        return []

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    scores = np.dot(chunk_embeddings, query_embedding)

    top_indices = scores.argsort()[-top_k:][::-1]

    results = []

    for idx in top_indices:
        chunk = document_chunks[idx].copy()
        chunk["score"] = float(scores[idx])
        results.append(chunk)

    return results

### Generate answer with Mistral

In [16]:
def generate_answer(query, retrieved_chunks, mode):
    context = "\n\n".join(
        [
            f"Source: {chunk['filename']}, Page {chunk['page_number']}\n{chunk['text']}"
            for chunk in retrieved_chunks
        ]
    )

    context = context[:3500]

    if mode == "Concise":
        instruction = "Answer briefly and directly."
    elif mode == "Detailed":
        instruction = "Answer in detail using the document context."
    else:
        instruction = "Answer clearly and mention the source page when possible."

    prompt = f"""
You are a document Q&A assistant.

Use ONLY the document context below to answer the user's question.
If the answer is not found in the context, say:
"I could not find the answer in the uploaded document."

Instruction:
{instruction}

Document Context:
{context}

User Question:
{query}

Answer:
"""

    answer = mistral_model(prompt, max_tokens=300)

    return answer

### Save chat history

In [17]:
chat_history = []

def save_chat_history():
    filename = "chat_history.json"

    with open(filename, "w", encoding="utf-8") as file:
        json.dump(chat_history, file, indent=4, ensure_ascii=False)

    return filename

### Main chatbot function

In [18]:
def answer_question(message, history, mode, top_k):
    if chunk_embeddings is None or len(document_chunks) == 0:
        return "Please upload and index at least one PDF first."

    retrieved_chunks = retrieve_relevant_chunks(message, top_k=int(top_k))

    if not retrieved_chunks:
        return "No relevant document chunks were found."

    answer = generate_answer(message, retrieved_chunks, mode)

    sources = "\n\n### Sources Used\n"

    for chunk in retrieved_chunks:
        sources += (
            f"- **{chunk['filename']}**, Page {chunk['page_number']}, "
            f"Chunk {chunk['chunk_index']}, Score: {chunk['score']:.3f}\n"
        )

    final_answer = answer + sources

    chat_history.append({
        "timestamp": datetime.now().isoformat(),
        "question": message,
        "answer": answer,
        "sources": retrieved_chunks
    })

    return final_answer

### Build Gradio interface

In [19]:
custom_css = """
.gradio-container {
    background: #f8fafc;
}

#title {
    text-align: center;
    color: #1e293b;
}

#subtitle {
    text-align: center;
    color: #475569;
}
"""

with gr.Blocks(theme=gr.themes.Soft(), css=custom_css) as demo:
    gr.Markdown(
        """
        # 📄 Mistral PDF Research Buddy
        """,
        elem_id="title"
    )

    gr.Markdown(
        """
        Upload one or more PDFs and ask questions about the document content.

        **Extra features added:** multiple PDF upload, answer mode selection, source display, top-k control, and chat history export.
        """,
        elem_id="subtitle"
    )

    with gr.Row():
        with gr.Column(scale=1):
            pdf_upload = gr.File(
                label="Upload PDF Document(s)",
                file_types=[".pdf"],
                file_count="multiple",
                type="filepath"
            )

            index_button = gr.Button("Build Document Index")

            index_status = gr.Textbox(
                label="Index Status",
                interactive=False
            )

            mode_dropdown = gr.Dropdown(
                choices=["Concise", "Detailed", "Cite Sources"],
                value="Cite Sources",
                label="Answer Mode"
            )

            top_k_slider = gr.Slider(
                minimum=1,
                maximum=5,
                value=3,
                step=1,
                label="Number of chunks to retrieve"
            )

            save_button = gr.Button("Save Chat History")
            saved_file = gr.File(label="Download Chat History")

        with gr.Column(scale=2):
            chatbot = gr.ChatInterface(
                fn=answer_question,
                additional_inputs=[mode_dropdown, top_k_slider],
                title="Ask Questions About Your PDFs"
            )

    index_button.click(
        fn=build_index,
        inputs=[pdf_upload],
        outputs=[index_status]
    )

    save_button.click(
        fn=save_chat_history,
        inputs=[],
        outputs=[saved_file]
    )

/tmp/ipykernel_439/1009057732.py:17: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=custom_css) as demo:


### Launch the chatbot

In [20]:
# demo.launch(share=True, debug=True)
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2cbd2f9ffaefe707e2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### Example questions to test

In [21]:
example_questions = [
    "What is this document about?",
    "What are the key dates mentioned?",
    "What part numbers are listed?",
    "Summarize the uploaded document.",
    "What compliance information is included?"
]

example_questions

['What is this document about?',
 'What are the key dates mentioned?',
 'What part numbers are listed?',
 'Summarize the uploaded document.',
 'What compliance information is included?']